# 04 · ETL — Violència a Cisjordània

**Fonts:**
- `data_6_Civilians_Israel.xlsx` — B'Tselem: palestins morts per civilians israelians (colons), 2000–2026
- `demolitions_1.xls` — B'Tselem: demolicions d'habitatges palestins, 2006–2026
- `west_bank_daily.csv` — Tech for Palestine / ONU: sèrie diària post-7O, 2023–2026

**Outputs** (a `data/clean/`):
- `fatalities_settlers_wb.csv` — morts per colons a Cisjordània
- `demolitions_wb.csv` — demolicions netes per localitat, districte i data
- `wb_daily_post7o.csv` — sèrie mensual agregada
- `wb_daily_post7o_daily.csv` — sèrie diària completa

---
## Rol en l'arquitectura

Aquests datasets alimenten les capes 5, 6 i 7 del mapa interactiu (notebook 06)  
i els gràfics d'anàlisi temporal (notebook 07).  
La geocodificació (coordenades per als mapes de punts) es fa al notebook 05.


## 1. Importació de llibreries

In [11]:
import pandas as pd
import os

print("Llibreries carregades correctament")

Llibreries carregades correctament


## 2. Configuració de paths

In [12]:
# Paths relatius des de la carpeta Notebook/
# Els datasets raw estan a ../../Datasets/raw/
RAW = "../Datasets/raw"
CLEAN = "data/clean"

FILE_FATALITIES  = f"{RAW}/data_6_Civilians_Israel.xlsx"
FILE_DEMOLITIONS = f"{RAW}/demolitions_1.xls"
FILE_DAILY       = f"{RAW}/west_bank_daily.csv"

OUT_FATALITIES   = f"{CLEAN}/fatalities_settlers_wb.csv"
OUT_DEMOLITIONS  = f"{CLEAN}/demolitions_wb.csv"
OUT_DAILY        = f"{CLEAN}/wb_daily_post7o.csv"
OUT_DAILY_DAILY  = f"{CLEAN}/wb_daily_post7o_daily.csv"

# Crear carpeta clean si no existeix
os.makedirs(CLEAN, exist_ok=True)

print("Paths configurats:")
print(f"  RAW:   {os.path.abspath(RAW)}")
print(f"  CLEAN: {os.path.abspath(CLEAN)}")

Paths configurats:
  RAW:   c:\Users\a-iba\OneDrive\Documentos\Sprint13\Datasets\raw
  CLEAN: c:\Users\a-iba\OneDrive\Documentos\Sprint13\Notebook\data\clean


---
## Secció A — Morts de palestins a mans de colons (2000–2026)

Font: B'Tselem  
134 registres individuals, tots `Killed By == 'Israeli civilians'`.  
Filtre: únicament `West Bank` (exclou Israel i Gaza Strip).


In [13]:
df_fat_raw = pd.read_excel(FILE_FATALITIES, sheet_name="Sheet1", engine="openpyxl")
print(f"Shape original: {df_fat_raw.shape}")
print(f"Regions: {df_fat_raw['Event location - Region'].value_counts().to_dict()}")

Shape original: (134, 16)
Regions: {'West Bank': 118, 'Israel': 12, 'Gaza Strip': 4}


In [14]:
# Filtre West Bank
df_fat = df_fat_raw[df_fat_raw["Event location - Region"] == "West Bank"].copy()

# Netejar espai inicial al nom de columna ' Killed By'
df_fat.columns = df_fat.columns.str.strip()

# Seleccionar i renombrar columnes rellevants
df_fat = df_fat[[
    "Name", "Date of event", "Date of death", "Age", "Gender",
    "Event location", "Event location - District",
    "Took part in the hostilities", "Type of injury", "Ammunition", "Notes"
]].rename(columns={
    "Date of event":                "date_event",
    "Date of death":                "date_death",
    "Event location":               "locality",
    "Event location - District":    "district",
    "Took part in the hostilities": "civilian_status",
    "Type of injury":               "injury_type",
})

# Dates i any
df_fat["date_event"] = pd.to_datetime(df_fat["date_event"], errors="coerce")
df_fat["date_death"] = pd.to_datetime(df_fat["date_death"], errors="coerce")
df_fat["year"]       = df_fat["date_event"].dt.year
df_fat["month"]      = df_fat["date_event"].dt.month
df_fat["Age"]        = pd.to_numeric(df_fat["Age"], errors="coerce")

df_fat = df_fat.sort_values("date_event").reset_index(drop=True)

print(f"Shape West Bank: {df_fat.shape}")
print(f"Rang: {df_fat['date_event'].min().date()} – {df_fat['date_event'].max().date()}")
print(f"\nDistrictes:")
print(df_fat["district"].value_counts())

Shape West Bank: (118, 13)
Rang: 2000-10-07 – 2026-04-22

Districtes:
district
Ramallah and al-Bira    36
Nablus                  31
Hebron                  22
Bethlehem               12
Salfit                   4
East Jerusalem           3
al-Quds                  3
Tulkarm                  3
Qalqiliya                2
Jericho                  1
Tubas                    1
Name: count, dtype: int64


In [15]:
# Comprovacions
print(f"Nuls per columna:\n{df_fat.isnull().sum()}")
print(f"\nDuplicats: {df_fat.duplicated().sum()}")
print(f"\nPer any:")
print(df_fat["year"].value_counts().sort_index())

Nuls per columna:
Name                0
date_event          0
date_death          0
Age                 2
Gender              0
locality            0
district            0
civilian_status     0
injury_type         0
Ammunition         43
Notes               0
year                0
month               0
dtype: int64

Duplicats: 0

Per any:
year
2000     5
2001     7
2002    11
2003     6
2004     2
2005     6
2008     3
2009     1
2010     2
2011     2
2014     1
2015     9
2016     4
2017     3
2018     4
2019     2
2021     2
2022     5
2023    14
2024     8
2025     9
2026    12
Name: count, dtype: int64


---
## Secció B — Demolicions d'habitatges palestins (2006–2026)

Font: B'Tselem  
5.465 registres. Tots ja filtrats a West Bank (`Area == 'west-bank'`).


In [16]:
df_dem_raw = pd.read_csv(FILE_DEMOLITIONS, encoding="utf-16", sep="\t")
print(f"Shape original: {df_dem_raw.shape}")
print(f"Columnes: {list(df_dem_raw.columns)}")

Shape original: (5465, 8)
Columnes: ['Date of Demolition', 'Locality', 'District', 'Area', 'Housing Units', 'People left Homeless', 'Minors left Homeless', 'Type Of Sturcture']


In [17]:
df_dem = df_dem_raw.rename(columns={
    "Date of Demolition":   "date",
    "Locality":             "locality",
    "District":             "district",
    "Area":                 "area",
    "Housing Units":        "housing_units",
    "People left Homeless": "people_homeless",
    "Minors left Homeless": "minors_homeless",
    "Type Of Sturcture":    "structure_type",  # manté l'error tipogràfic de la font
}).copy()

# Dates
df_dem["date"]  = pd.to_datetime(df_dem["date"], errors="coerce")
df_dem["year"]  = df_dem["date"].dt.year
df_dem["month"] = df_dem["date"].dt.month

# Numèrics
for col in ["housing_units", "people_homeless", "minors_homeless"]:
    df_dem[col] = pd.to_numeric(df_dem[col], errors="coerce").fillna(0).astype(int)

df_dem["structure_type"] = df_dem["structure_type"].str.strip().str.lower()
df_dem = df_dem.sort_values("date").reset_index(drop=True)

print(f"Shape net: {df_dem.shape}")
print(f"Rang: {df_dem['date'].min().date()} – {df_dem['date'].max().date()}")
print(f"\nDistrictes:")
print(df_dem["district"].value_counts())

Shape net: (5465, 10)
Rang: 2006-01-04 – 2026-05-31

Districtes:
district
Hebron                  1134
al-Quds                  939
Tubas                    772
Jericho                  665
Nablus                   605
Ramallah and al-Bira     456
Bethlehem                320
Jenin                    207
Salfit                   148
Qalqiliya                133
Tulkarm                   77
East Jerusalem             3
Israel                     2
Name: count, dtype: int64


In [18]:
# Comprovacions i resum
print(f"Nuls per columna:\n{df_dem.isnull().sum()}")
print(f"\nDuplicats: {df_dem.duplicated().sum()}")
print(f"\nResum numèric:")
print(f"  Total housing units demolides: {df_dem['housing_units'].sum():,}")
print(f"  Total persones desplaçades:    {df_dem['people_homeless'].sum():,}")
print(f"  Total menors desplaçats:       {df_dem['minors_homeless'].sum():,}")
print(f"\nPer any (últims 5):")
print(df_dem.groupby("year")[["housing_units","people_homeless"]].sum().tail(5))

Nuls per columna:
date               0
locality           3
district           4
area               0
housing_units      0
people_homeless    0
minors_homeless    0
structure_type     0
year               0
month              0
dtype: int64

Duplicats: 1333

Resum numèric:
  Total housing units demolides: 8,206
  Total persones desplaçades:    11,233
  Total menors desplaçats:       5,671

Per any (últims 5):
      housing_units  people_homeless
year                                
2022            784              500
2023            554              401
2024            871              953
2025           1195             1116
2026            474              399


---
## Secció C — Sèrie diària post-7O (2023–2026)

Font: Tech for Palestine / ONU  
1.012 registres diaris des del 7-O (2023-10-07).  
Usem `killed_cum`, `injured_cum` i `settler_attacks_cum` (sense NaN).  
Les columnes `verified.*` tenen NaN per retard en la verificació de l'ONU.


In [19]:
df_wb_raw = pd.read_csv(FILE_DAILY)
print(f"Shape: {df_wb_raw.shape}")
print(f"Rang: {df_wb_raw['report_date'].min()} – {df_wb_raw['report_date'].max()}")
print(f"\nNuls per columna:")
print(df_wb_raw.isnull().sum())

Shape: (1012, 15)
Rang: 2023-10-07 – 2026-07-14

Nuls per columna:
report_date                        0
verified.killed                  609
verified.killed_cum              608
verified.injured                 623
verified.injured_cum             621
verified.killed_children         609
verified.killed_children_cum     608
verified.injured_children        623
verified.injured_children_cum    621
killed_cum                         0
killed_children_cum                0
injured_cum                        0
injured_children_cum               0
settler_attacks_cum                0
flash_source                       0
dtype: int64


In [20]:
df_wb = df_wb_raw.copy()
df_wb["date"]  = pd.to_datetime(df_wb["report_date"], errors="coerce")
df_wb["year"]  = df_wb["date"].dt.year
df_wb["month"] = df_wb["date"].dt.month

df_wb = df_wb[[
    "date", "year", "month",
    "killed_cum", "killed_children_cum",
    "injured_cum", "injured_children_cum",
    "settler_attacks_cum",
    "verified.killed", "verified.injured",
]].rename(columns={
    "killed_cum":           "deaths_cum",
    "killed_children_cum":  "deaths_children_cum",
    "injured_children_cum": "injured_children_cum",
    "settler_attacks_cum":  "settler_attacks_cum",
    "verified.killed":      "verified_killed",
    "verified.injured":     "verified_injured",
}).sort_values("date").reset_index(drop=True)

print(f"Shape net: {df_wb.shape}")
print(f"\nÚltima fila (acumulats finals):")
print(df_wb.iloc[-1][["date","deaths_cum","injured_cum","settler_attacks_cum"]])

Shape net: (1012, 10)

Última fila (acumulats finals):
date                   2026-07-14 00:00:00
deaths_cum                            1094
injured_cum                          11241
settler_attacks_cum                   4207
Name: 1011, dtype: object


In [21]:
# Agregació mensual
df_monthly = df_wb.groupby(["year","month"]).agg(
    deaths_cum_end          = ("deaths_cum",           "last"),
    injured_cum_end         = ("injured_cum",           "last"),
    settler_attacks_cum_end = ("settler_attacks_cum",   "last"),
    verified_killed_month   = ("verified_killed",       "sum"),
    verified_injured_month  = ("verified_injured",      "sum"),
).reset_index()

# Valors mensuals (no acumulats)
df_monthly["deaths_month"]          = df_monthly["deaths_cum_end"].diff().fillna(df_monthly["deaths_cum_end"])
df_monthly["settler_attacks_month"] = df_monthly["settler_attacks_cum_end"].diff().fillna(df_monthly["settler_attacks_cum_end"])
df_monthly["date"] = pd.to_datetime(df_monthly[["year","month"]].assign(day=1))

print(f"Shape mensual: {df_monthly.shape}")
print(f"\nMesos amb més morts:")
print(df_monthly.nlargest(5,"deaths_month")[["date","deaths_month","settler_attacks_month"]])

Shape mensual: (34, 10)

Mesos amb més morts:
         date  deaths_month  settler_attacks_month
0  2023-10-01         123.0                  178.0
1  2023-11-01         118.0                  121.0
11 2024-09-01          77.0                  122.0
2  2023-12-01          66.0                   71.0
3  2024-01-01          63.0                  124.0


In [22]:
# Comprovació monotonicitat acumulats
assert df_wb["deaths_cum"].is_monotonic_increasing, "ERROR: deaths_cum no és monòton"
assert df_wb["settler_attacks_cum"].is_monotonic_increasing, "ERROR: settler_attacks_cum no és monòton"
print("✓ Sèries acumulades coherents")
print(f"\nResum final:")
print(f"  Morts totals (WB):          {df_wb['deaths_cum'].iloc[-1]:,}")
print(f"  Ferits totals (WB):         {df_wb['injured_cum'].iloc[-1]:,}")
print(f"  Atacs colons acumulats (WB): {df_wb['settler_attacks_cum'].iloc[-1]:,}")

✓ Sèries acumulades coherents

Resum final:
  Morts totals (WB):          1,094
  Ferits totals (WB):         11,241
  Atacs colons acumulats (WB): 4,207


## 3. Exportació

In [23]:
# A — Fatalities per colons
df_fat.to_csv(OUT_FATALITIES, index=False)
print(f"✓ {OUT_FATALITIES}")
print(f"  {len(df_fat)} registres | {df_fat['year'].min()}–{df_fat['year'].max()}")

# B — Demolicions
df_dem.to_csv(OUT_DEMOLITIONS, index=False)
print(f"\n✓ {OUT_DEMOLITIONS}")
print(f"  {len(df_dem)} registres | {df_dem['year'].min()}–{df_dem['year'].max()}")
print(f"  {df_dem['housing_units'].sum():,} unitats demolides | {df_dem['people_homeless'].sum():,} persones desplaçades")

# C — West Bank Daily (mensual + diària)
df_monthly.to_csv(OUT_DAILY, index=False)
df_wb.to_csv(OUT_DAILY_DAILY, index=False)
print(f"\n✓ {OUT_DAILY} (mensual: {len(df_monthly)} mesos)")
print(f"✓ {OUT_DAILY_DAILY} (diària: {len(df_wb)} dies)")
print(f"\nTots els datasets de violència exportats.")
print(f"Proper pas: notebook 05 — capes geogràfiques del mapa.")

✓ data/clean/fatalities_settlers_wb.csv
  118 registres | 2000–2026

✓ data/clean/demolitions_wb.csv
  5465 registres | 2006–2026
  8,206 unitats demolides | 11,233 persones desplaçades

✓ data/clean/wb_daily_post7o.csv (mensual: 34 mesos)
✓ data/clean/wb_daily_post7o_daily.csv (diària: 1012 dies)

Tots els datasets de violència exportats.
Proper pas: notebook 05 — capes geogràfiques del mapa.
